# Routerset Materialized Dataset Inspection

This notebook opens the canonical materialized routerset export, checks the manifest and file invariants, and visualizes representative `8x256x256` tiles.

In [ ]:
from __future__ import annotations

import json
import os
import random
from collections import Counter, defaultdict
from pathlib import Path
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np


def default_dataset_root() -> Path:
    override = os.environ.get('ROUTERSET_MATERIALIZED_ROOT')
    if override:
        return Path(override).expanduser().resolve()
    candidates = [
        Path.cwd() / 'outputs' / 'routerset' / 'materialized_256',
        Path.cwd().parent / 'outputs' / 'routerset' / 'materialized_256',
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    return candidates[0].resolve()


DATASET_ROOT = default_dataset_root()
MANIFEST_PATH = DATASET_ROOT / 'manifest_256.jsonl'
SUMMARY_PATH = DATASET_ROOT / 'materialization_summary.json'
REPORT_PATH = DATASET_ROOT / 'dataset_report.json'

print('DATASET_ROOT =', DATASET_ROOT)
assert MANIFEST_PATH.exists(), f'Missing materialized manifest: {MANIFEST_PATH}'
assert SUMMARY_PATH.exists(), f'Missing materialization summary: {SUMMARY_PATH}'
assert REPORT_PATH.exists(), f'Missing dataset report: {REPORT_PATH}'

In [ ]:
rows = [json.loads(line) for line in MANIFEST_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
dataset_report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))

print(f'manifest rows: {len(rows):,}')
print('experts:', ', '.join(summary['experts']))
print('materialized files:', f"{summary['materialized_unique_file_count']:,}")
print('materialized size (GiB):', round(summary['materialized_total_bytes'] / (1024 ** 3), 2))
print('corrections applied:')
for item in summary['corrections_applied']:
    print(' -', item)

In [ ]:
rows_by_dataset = Counter(row['source_dataset'] for row in rows)
rows_by_split = Counter((row['source_dataset'], row['source_split']) for row in rows)
print('Rows by dataset:')
pprint(dict(sorted(rows_by_dataset.items())))
print('\nRows by dataset/split:')
for key, value in sorted(rows_by_split.items()):
    print(f'{key}: {value}')

print('\nDataset report summary keys:')
print(sorted(dataset_report.keys()))

In [ ]:
expected_shape = '8x256x256'
shape_violations = {}
for split_name in ('train', 'validation'):
    split_shapes = dataset_report[split_name]['training_shapes_by_expert']
    for dataset, shapes in split_shapes.items():
        if list(shapes.keys()) != [expected_shape]:
            shape_violations.setdefault(split_name, {})[dataset] = shapes

print('shape_violations:', shape_violations or 'none')

header_check_rows = random.sample(rows, min(len(rows), 256))
dtype_violations = []
for row in header_check_rows:
    path = Path(row['materialized_image_path'])
    array = np.load(path, mmap_mode='r')
    if array.shape != (8, 256, 256) or array.dtype != np.float32:
        dtype_violations.append({'path': str(path), 'shape': tuple(array.shape), 'dtype': str(array.dtype)})

print('sampled_header_violations:', len(dtype_violations))
if dtype_violations:
    pprint(dtype_violations[:10])

In [ ]:
def rgb_from_tile(tile: np.ndarray) -> np.ndarray:
    if tile.shape[0] >= 3:
        rgb = tile[[2, 1, 0]].transpose(1, 2, 0)
    else:
        rgb = np.repeat(tile[:1].transpose(1, 2, 0), 3, axis=2)
    lo, hi = np.quantile(rgb, [0.02, 0.98])
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        hi = max(float(np.max(rgb)), 1.0)
        lo = min(float(np.min(rgb)), 0.0)
    return np.clip((rgb - lo) / max(hi - lo, 1e-6), 0.0, 1.0)


def sample_stats(row: dict) -> dict:
    array = np.load(row['materialized_image_path'])
    return {
        'shape': tuple(array.shape),
        'dtype': str(array.dtype),
        'min': float(array.min()),
        'max': float(array.max()),
        'mean': float(array.mean()),
        'zero_fraction': float(np.mean(array == 0)),
    }


expert_order = ['anomaly_detection', 'burned_area', 'fire', 'lc', 'roads', 'worldfloods']
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, dataset in zip(axes.ravel(), expert_order):
    row = next(row for row in rows if row['source_dataset'] == dataset)
    tile = np.load(row['materialized_image_path'])
    stats = sample_stats(row)
    ax.imshow(rgb_from_tile(tile))
    ax.set_title(f"{dataset}\nzero={stats['zero_fraction']:.3f} mean={stats['mean']:.4f}")
    ax.axis('off')
plt.tight_layout()

In [ ]:
random.seed(0)
zero_fraction_by_dataset = defaultdict(list)
sample_limit = 64
for dataset in sorted({row['source_dataset'] for row in rows}):
    dataset_rows = [row for row in rows if row['source_dataset'] == dataset]
    if len(dataset_rows) > sample_limit:
        dataset_rows = random.sample(dataset_rows, sample_limit)
    for row in dataset_rows:
        array = np.load(row['materialized_image_path'], mmap_mode='r')
        zero_fraction_by_dataset[dataset].append(float(np.mean(array == 0)))

fig, ax = plt.subplots(figsize=(12, 4))
datasets = list(sorted(zero_fraction_by_dataset))
ax.boxplot([zero_fraction_by_dataset[dataset] for dataset in datasets], tick_labels=datasets)
ax.set_ylabel('sampled zero fraction')
ax.set_title('Sampled zero-fraction distribution by expert')
ax.grid(alpha=0.2)
plt.xticks(rotation=20)
plt.show()

print('Sampled zero-fraction medians:')
for dataset in datasets:
    values = zero_fraction_by_dataset[dataset]
    print(dataset, round(float(np.median(values)), 4))

## Notes

- The exported dataset is expected to contain only concrete `.npy` files referenced by `manifest_256.jsonl`.
- `roads` and `lc` are already normalized into the Phi2FM student-compatible 8-channel layout before export.
- High zero-fraction for `roads`, `lc`, and `burned_area` reflects canonical padding to `256x256`, not a broken display pipeline in this notebook.